In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [6]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# A. Membaca ketiga sumber data dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Jumlah Baris -> Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Jumlah Baris -> Transaksi: 4000 | Cabang: 3 | Menu: 15


In [12]:
from pyspark.sql.functions import col

# B. Gabungkan ketiga tabel & tambahkan kolom total_penjualan (qty * harga)
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as spark_sum, row_number

# C. Top-2 menu terlaris berdasarkan total_penjualan di setiap cabang
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [14]:
# D. Spark SQL murni: Total penjualan per kepala_barista
df_gabungan.createOrReplaceTempView("tugas7_gabungan")

kontribusi_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM tugas7_gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [17]:
# E. Simpan data gabungan ke HDFS sebagai Parquet terpartisi berdasarkan kategori_menu
# Catatan: ganti 'mahasiswa' dengan username HDFS milikmu jika berbeda
path_output = "/user/mahasiswa/tugas7/processed/laporan"

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(f"hdfs://localhost:9000{path_output}")

# Verifikasi dengan membaca kembali dan menghitung jumlah baris
df_verifikasi = spark.read.parquet(f"hdfs://localhost:9000{path_output}")
print("Verifikasi simpan Parquet berhasil. Total baris data:", df_verifikasi.count())

Verifikasi simpan Parquet berhasil. Total baris data: 4000


In [16]:
from pyspark.sql.functions import avg, count

# F. Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi)
ringkasan_eksekutif = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan hasil analisis pipeline pemrosesan Big Data pada kedai "Kopi Nusantara", cabang yang menunjukkan kinerja terbaik secara keseluruhan adalah cabang dengan total nilai penjualan triwulan dan volume transaksi tertinggi. Keberhasilan cabang terbaik ini didorong oleh tingginya frekuensi kunjungan pelanggan serta tingginya angka rata-rata nilai transaksi per pembeli, yang dikepalai oleh kinerja barista yang efektif dalam menjaga kualitas layanan.

Sebaliknya, pada cabang dengan kinerja terlemah, diperlukan strategi intervensi bisnis yang lebih terarah untuk mendongkrak omzet. Berdasarkan temuan analisis Window Function pada bagian C, menu yang menduduki peringkat teratas (Top-2 terlaris) di cabang lain harus dipromosikan lebih gencar di cabang terlemah ini. Alasan utamanya adalah menu-menu terlaris tersebut telah terbukti memiliki product-market fit dan daya tarik yang kuat bagi preferensi konsumen secara umum. Dengan menggencarkan promosi bundling, diskon khusus, serta penempatan menu favorit tersebut pada papan highlight kasir di cabang terlemah, kedai dapat menarik minat pelanggan baru sekaligus meningkatkan nilai rata-rata transaksi (average basket size) untuk menyeimbangkan performa antar cabang.